# The engine, end to end

How a strategy goes from an idea to a run you can read, on your own data: load the bars, define a feature, write a signal strategy, run it, read its metrics, explain one of its trades, plot that trade's path, measure how much its fills depend on the data's fidelity, and compare a few of its variants. `docs/engine-guide.md` is the reference for every option the cells use, and `docs/spec/fill-semantics.md` holds the rules behind every fill.

Pick the `.venv` kernel and run all cells. The notebook reads the store in `FEFAT_DATA_DIR` and changes nothing in it. It saves its runs, and the engine's cache, under `FEFAT_HOME`, and for the fidelity check it generates a synthetic year of ES with ticks into `FEFAT_HOME`'s cache folder, about 1.4 GB, which may be deleted at any time. Your store holds MES's 1-minute bars and no ticks, so the strategy decides on 5-minute closes and its orders fill on 1-minute bars. Above each cell is what it should print on this PC.

## Setup

The product and the windows: MES when your store holds its 1-minute bars, else the 1-minute bars with the most contracts; the run spans the last five years the store holds, and the variants its last year.

Expected on this PC: `MES from databento: 2021-09-24 to 2026-09-23`.

In [ ]:
import json
import os
import subprocess
import sys
import time
import warnings
from contextlib import contextmanager
from datetime import date, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from IPython.display import display

import fefat as ff
from fefat.backtest import FidelityCheck, RunConfig
from fefat.config import resolve_paths
from fefat.market.calendar import counted
from fefat.market.saved_instruments import instrument_table

REPOSITORY = Path(ff.__file__).resolve().parents[4]
paths = resolve_paths(os.environ)
table = instrument_table(paths.data_dir)
listed = ff.market.datasets().filter(pl.col("schema") == "ohlcv-1m")
if listed.is_empty():
    raise ValueError(
        f"Expected 1-minute bars of at least one product in {paths.data_dir}, got none: "
        "import your files first (docs/data-guide.md)."
    )
chosen = listed.sort(
    pl.col("root") != "MES", "contracts", descending=[False, True], maintain_order=True
).row(0, named=True)
ROOT, PROVIDER, END = chosen["root"], chosen["provider"], chosen["last_trade_date"]
START = max(chosen["first_trade_date"], END - timedelta(days=5 * 365))
VARIANTS_START = max(START, END - timedelta(days=365))
CONFIG = RunConfig(fidelity="1m", provider=PROVIDER)
print(f"{ROOT} from {PROVIDER}: {START} to {END}")

## Load the bars

`ff.market.load` builds bars of any timeframe from the store's finest bars: here the 5-minute bars the strategy decides on, spliced into one continuous series across the rolls and adjusted from the front backwards, each bar keeping its own contract and its raw close. A run loads its bars the same way by itself; this cell only shows what it reads. `docs/data-guide.md` explains every column.

Expected on this PC: `Loaded 353,934 five-minute bars of 21 contracts over 1,257 sessions`, and the columns of a load.

In [ ]:
bars = ff.market.load(ROOT, "5m", START, END, fidelity="1m", provider=PROVIDER).collect()
contracts = bars.get_column("contract").n_unique()
sessions_loaded = bars.get_column("trade_date").n_unique()
print(
    f"Loaded {bars.height:,} five-minute bars of {counted(contracts, 'contract')} over "
    f"{counted(sessions_loaded, 'session')}"
)
print(f"Columns: {', '.join(bars.columns)}")

## A feature, inline

Until Phase 3 brings `@ff.feature` and its leak detector, a feature is a Polars expression over the bars, which the run evaluates as a new column. Each must look only backwards (D-12): `shift(1)` makes `highest` the highest of the 12 closes before the bar, so that a close never compares with itself, and a rolling mean ends at the bar it is on. The ATR here is the mean range of the last 12 bars, in the bars' price units, which a stop or a target in ATRs reads. A feature never uses `shift(-1)`, a centred window, or a statistic of the whole sample, which would let a bar see its future.

Expected on this PC: about 15 % of the bars closing above the highest of the 12 before and 13 % below the lowest, and a median ATR of about 12 ticks.

In [ ]:
close = pl.col("close")
FEATURES = {
    "highest": close.rolling_max(12).shift(1),
    "lowest": close.rolling_min(12).shift(1),
    "atr": (pl.col("high") - pl.col("low")).rolling_mean(12),
}
tick = int(table.get(ROOT).tick_size.scaleb(9))
shares = (
    bars.with_columns(**FEATURES)
    .select(
        above=(close > pl.col("highest")).mean(),
        below=(close < pl.col("lowest")).mean(),
        atr_ticks=(pl.col("atr") / tick).median(),
    )
    .row(0, named=True)
)
print(
    f"Closes above the highest of the 12 before: {shares['above']:.1%}, below the lowest: "
    f"{shares['below']:.1%}; the median ATR is {shares['atr_ticks']:.1f} ticks"
)

## The strategy

A breakout both ways: long on a 5-minute close above the highest of the 12 closes before it, short on one below the lowest, entering between 08:30 and 15:00 Chicago time and flat from 15:05. Each entry is a market order of one lot with a stop 1.5 ATRs away and a target at twice the risk; one position at a time, and an opposite signal is ignored while a trade is open. `strategy.to_spec()` is the document the engine reads, which every run keeps in its `run.json`.

Expected: the specification as JSON, its execution template naming the market entry, the ATR stop, the target in R, the time rules, and the fixed size.

In [ ]:
def breakout(root: str, stop_atr: float, target_r: float) -> ff.SignalStrategy:
    """Long on a close above the highest of the 12 before, short below the lowest, entering in
    the day session, with a stop `stop_atr` ATRs away and a target `target_r` times the risk."""
    zone = table.get(root).session_calendar().hours.time_zone
    return ff.SignalStrategy(
        instrument=root,
        timeframe="5m",
        features=FEATURES,
        long_entry=close > pl.col("highest"),
        short_entry=close < pl.col("lowest"),
        execution=ff.Execution(
            entry=ff.Market(),
            stop=ff.Stop(atr=stop_atr),
            targets=[ff.Target(r=target_r)],
            time=ff.TimeRules(tz=zone, windows=[("08:30", "15:00")], flat_by="15:05"),
            sizing=ff.Fixed(1),
        ),
    )


strategy = breakout(ROOT, stop_atr=1.5, target_r=2.0)
print(json.dumps(strategy.to_spec().document(), indent=2))

## The run

`ff.backtest.run` runs the strategy over the five years in the Rust engine. Each trade date fills on its finest data, here MES's 1-minute bars, each bar replayed as its open, the extreme against the position, the extreme in its favour, and its close; with one tick of slippage on market and stop orders, 100 ms from a decision to its orders, and the fees of `retail-default`. The run is saved under its run key in `FEFAT_HOME`, with each trade's path, and run again at one more tick of slippage. A warning names the fee table when its first schedule starts after your first trade date (F-FEE-01).

Running the cell again reads the saved run back in a fraction of the time: to time a fresh run, delete the folder it prints first.

Expected on this PC: `Ran in` about 18 s the first time, while the engine's cache of the months it lacked was built, and about 9 s on a fresh run once it is; the run's key and folder, and one fidelity era, `1m`, over 1,257 trade dates; about 4.5 s when the cell reads the saved run back, most of it spent building the signal frame that the run key hashes.

In [ ]:
started = time.perf_counter()
run = ff.backtest.run(strategy, START, END, CONFIG)
print(f"Ran in {time.perf_counter() - started:.1f} s: run {run.hash[:12]} in {run.folder}")
for era in run.record.fidelity_eras:
    dates = counted(era.trade_dates, "trade date")
    print(f"Fidelity {era.fidelity}: {dates} from {era.first} to {era.last}")

## Its metrics

`run.report()` prints the run's fidelity eras, its metrics with the trades split by the fidelity of their entry, the profit and loss at one more tick of slippage, the robustness summary of the backtest-expert skill with its verdict and red flags, and the fidelity check, which a run on 1-minute bars alone cannot have. `run.metrics` holds the same numbers for code; a ratio that cannot be worked out, such as a profit factor without a losing trade, is `None`.

Expected on this PC: the report, then one line with the trades, 7,753 of them, the net profit and loss, the profit factor, the expectancy per trade, and the maximum drawdown. A breakout this simple loses after costs, and the robustness summary says so; here the reading matters, not the strategy.

In [ ]:
run.report()
METRICS = run.metrics


def shown(value: float | None, form: str) -> str:
    return "none" if value is None else format(value, form)


statistics = METRICS.trades
print(
    f"\nTrades: {statistics.trades}, net P&L {METRICS.pnl:,.2f}, profit factor "
    f"{shown(statistics.profit_factor, '.2f')}, expectancy {shown(statistics.expectancy, ',.2f')}, "
    f"max drawdown {METRICS.drawdown.max_drawdown:,.2f}"
)

## One trade explained

`run.explain` replays one trade through the reference engine, FEFAT's slow, readable implementation of the fill specification, on the run's own bars, and checks that it fills the trade as the Rust engine did. It takes the trade's `entry_order`, from `run.trades`. Each line is an event with the rule behind it in brackets; each fill names the rule that priced it and its slippage in ticks, and a fill on a bar shows that bar's open, high, low, and close.

Expected: the first trade stopped out at a loss within one contract, from the decision that placed its entry to its stop's fill (F-STP-04 on 1-minute bars), and its target cancelled (F-OCO-01).

In [ ]:
numbered = run.trades.with_row_index("trade")
stopped = numbered.filter(pl.col("exit_reason") == "stop", pl.col("pnl") < 0, pl.col("rolls") == 0)
if stopped.is_empty():
    raise ValueError(f"Expected a trade stopped out at a loss in run {run.hash[:12]}, got none")
TRADE = stopped.row(0, named=True)
EXPLANATION = run.explain(TRADE["entry_order"])

## Its path

Every saved run keeps each trade's compact path in `trade_paths.parquet`: its fills, each new best price and the worst since, and the mark at every whole minute, which the prop simulator will read. The plot draws the trade above in ticks from its entry, so that it shows no price of your data: the path, its fills, the entry at 0, the trade's MAE, the furthest the price went against it, and its exit.

Expected on this PC: the plot, and `The path of trade 1: 65 points`, then the points by kind.

In [ ]:
path = run.trade_paths.filter(pl.col("trade") == TRADE["trade"])
sign = 1 if TRADE["side"] == "long" else -1
points = path.with_columns(
    at=pl.from_epoch("ts", time_unit="ns"), ticks=(pl.col("price") - TRADE["entry_price"]) / tick
)
fills = points.filter(pl.col("kind") == "fill")
figure, axes = plt.subplots(figsize=(10, 4))
axes.plot(points.get_column("at"), points.get_column("ticks"), linewidth=1, label="path")
axes.scatter(fills.get_column("at"), fills.get_column("ticks"), color="black", label="fills")
axes.axhline(0, color="grey", linestyle=":", label="entry")
mae = f"MAE, {TRADE['mae_ticks']} ticks"
axes.axhline(-sign * TRADE["mae_ticks"], color="red", linestyle="--", label=mae)
exit_ticks = (TRADE["exit_price"] - TRADE["entry_price"]) / tick
axes.axhline(exit_ticks, color="green", linestyle=":", label="exit")
axes.set_ylabel("ticks from the entry")
axes.set_title(f"Trade {TRADE['trade']}, path fidelity {TRADE['path_fidelity']}")
axes.legend(loc="best")
kinds = path.group_by("kind", maintain_order=True).len()
print(
    f"The path of trade {TRADE['trade']}: {counted(path.height, 'point')}, "
    + ", ".join(f"{count} {kind}" for kind, count in kinds.iter_rows())
)

## How much the fills depend on the data

A run whose trade dates are finer than a minute is also run again at 1-second and at 1-minute fidelity, and `ff.backtest.fidelity_check(run)` gives the differences: the error bar of a backtest on bars. Your store holds no ticks of MES, so the check runs here on a synthetic year of ES generated by seed, with trades, 1-second, and 1-minute bars in every session, the same year the engine's benchmarks run on (`docs/validation/benchmarks.md`). The year holds one contract, ESH2026, past its real expiry, so the run reads that contract unspliced (`RunConfig(contract=...)`) rather than a continuous series, whose roll calendar ends at the expiry. The first time, generating the year takes about two and a half minutes; later the cell reuses it. While the cell runs, `FEFAT_DATA_DIR` points at the year's store, then back at yours.

Expected on this PC: `252 sessions from 2026-01-02 to 2026-12-31 ready`, `Ran with its check in` about 13 s the first time and 5 s later, `Baseline on trades: 1956 trades, net P&L -62,327.44`, `At 1s: -2 trades, net P&L +2,973.98`, and `At 1m: -2 trades, net P&L +9,861.48`: the 1-minute fills come out about 16 % better, the error bar of this strategy's days on bars. A warning says that the ATR lies below one tick on a few entry rows of the year's quiet stretches, where the stop then lies one tick away (F-RND-01). The generated tape is a random walk, so the strategy loses: here the size of the differences matters, not the result.

In [ ]:
@contextmanager
def store_at(folder: Path):
    """`FEFAT_DATA_DIR` pointed at `folder` while the block runs, then back at yours."""
    yours = os.environ.get("FEFAT_DATA_DIR")
    os.environ["FEFAT_DATA_DIR"] = str(folder)
    try:
        yield
    finally:
        if yours is None:
            os.environ.pop("FEFAT_DATA_DIR")
        else:
            os.environ["FEFAT_DATA_DIR"] = yours


FIRST = date(2026, 1, 2)
year_sessions = table.sessions_of("ES", paths.cache_dir).between(FIRST, date(2026, 12, 31))
year_sessions = year_sessions.get_column("trade_date").to_list()
year_sessions = year_sessions[: int(os.environ.get("FEFAT_CHECK_DAYS", len(year_sessions)))]
LAST = year_sessions[-1]
year_store = paths.cache_dir / f"synthetic-es-{FIRST}-{LAST}" / "data"
if not (year_store / "synthetic" / "MANIFEST.json").is_file():
    generator = REPOSITORY / "scripts" / "gen_fixtures.py"
    dates = ["--dates", f"{FIRST}..{LAST}", "--contracts", "ESH2026", "--planted-gaps", "3"]
    command = [sys.executable, str(generator), "--out", str(year_store / "synthetic"), *dates]
    generated = subprocess.run(command, capture_output=True, text=True, check=False)
    if generated.returncode != 0:
        raise RuntimeError(f"Expected gen_fixtures.py to pass, got: {generated.stderr}")
print(f"{counted(len(year_sessions), 'session')} from {FIRST} to {LAST} ready")
started = time.perf_counter()
with store_at(year_store):
    one_contract = RunConfig(contract="ESH2026")
    year_run = ff.backtest.run(
        breakout("ES", stop_atr=1.5, target_r=2.0), FIRST, LAST, one_contract
    )
    CHECK = ff.backtest.fidelity_check(year_run)
print(f"Ran with its check in {time.perf_counter() - started:.0f} s")
if not isinstance(CHECK, FidelityCheck):
    raise ValueError(f"Expected a fidelity check of the generated sessions, got {CHECK!r}")
baseline = CHECK.baseline
print(f"Baseline on {baseline.fidelity}: {baseline.trades} trades, net P&L {baseline.pnl:,.2f}")
for rerun in filter(None, (CHECK.at_one_second, CHECK.at_one_minute)):
    print(
        f"At {rerun.outcome.fidelity}: {rerun.trades_delta:+d} trades, "
        f"net P&L {rerun.pnl_delta:+,.2f}"
    )

## A few variants

Nine runs of the strategy over the last year, the stop at 1, 1.5, and 2 ATRs and the target at 1.5, 2, and 3 times the risk, each saved like the run above. Look for a plateau, a region where neighbouring parameters do about as well, rather than for the single best cell, which is the likeliest to be luck: the backtest-expert skill's "seek plateaus, not peaks". Phase 3's research tools run such sweeps at scale; until then, a loop over `ff.backtest.run` does it. Each run would repeat the fee table's warning, which the run above showed, so the loop silences that one warning and no other.

Expected on this PC: `Variants: 9 runs in` about 45 s the first time, then a table of nine rows, one per variant, with its trades, net profit and loss, profit factor, and maximum drawdown.

In [ ]:
started = time.perf_counter()
variants = []
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*F-FEE-01")
    for stop_atr in (1.0, 1.5, 2.0):
        for target_r in (1.5, 2.0, 3.0):
            strategy_variant = breakout(ROOT, stop_atr, target_r)
            metrics = ff.backtest.run(strategy_variant, VARIANTS_START, END, CONFIG).metrics
            variants.append(
                {
                    "stop_atr": stop_atr,
                    "target_r": target_r,
                    "trades": metrics.trades.trades,
                    "pnl": float(metrics.pnl),
                    "profit_factor": metrics.trades.profit_factor,
                    "max_drawdown": float(metrics.drawdown.max_drawdown),
                }
            )
VARIANTS = pl.DataFrame(variants)
print(f"Variants: {VARIANTS.height} runs in {time.perf_counter() - started:.0f} s")
display(VARIANTS)

## Your check

Run the tutorial. Note the time for a 5-year run on your machine; it should be seconds, not minutes.

The run cell prints it as `Ran in … s`. Time it on a fresh run: a second run of the cell reads the saved run back, so delete the folder the cell prints first. The first run of a product also builds the engine's cache of its months, which later runs reuse. On this PC the five years of MES ran in 18.2 s the first time and in about 9 s on later fresh runs with the cache built, most of it spent checking that cache against the store, not in the engine.